<a href="https://colab.research.google.com/github/hjiwoong/ML/blob/main/Chapter08_%ED%85%8D%EC%8A%A4%ED%8A%B8%EB%B6%84%EC%84%9D_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 텍스트 전처리 - 텍스트 정규화
- NLTK(Natural Language Toolkit)
- punkt: 문장, 단어 토큰화에 사용하는 사전학습 모델
- 약어(Mr., Dr.)와 일반 마침표를 구분해 문장을 올바르게 분리
- punkt_tab : 최신 NLTK에서 punkt 대신 사용하는 토큰화 모델
- stopwords: 영어 불용어(Stop Words) 목록 (약 179개),
            분석에 의미 없는 빈출 단어: the, is, are, you, we 등,
            제거하면 중요 단어에 집중 가능. 피처 수 감소
- wordnet: 영어 어휘 사전 (단어 간 의미 관계 포함),
          Lemmatization(표제어 추출, 어근 추출)시 원형 단어 조회에 사용,
          표제어(Lemma)는 단어의 사전적 기본형 원형 의미
- averaged_perceptron_tagger: 품사 태깅(POS Tagging) 모델.
                              각 단어의 품사(명사/동사/형용사 등)를 자동으로 판별
- omw-1.4: Open Multilingual Wordnet(다국어 Wordnet), wordnet 사용시 함께 필요

In [ ]:
# NLTK 필수 리소스 다운로드, 처음 한 번만 다운로드하면 됨, 다운로드 위치: -/nltk_data/
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

### 문장 토큰화(Sentence Tokenization)

[토큰화(Tokenization)란?]
- 텍스트를 의미 있는 최소 단위(토큰)로 분리하는 전처리 과정
- NLP의 첫 번째 단계: 원시 텍스트 -> 처리 가능한 단위로 변환
- 자연어 처리 (Natural Language Processing)

[sent_tokenize 동작 원리]
- punkt 모델 기반: 마침표(.) 뒤를 무조건 문장 끝으로 보지 않음
- Mr., Dr., etc. 같은 약어와 일반 마침표를 구분해 올바르게 분리
- 마침표·느낌표·물음표를 기준으로 문장 분리

In [ ]:
from nltk import sent_tokenize, word_tokenize
text_sample = ('The Matrix is everywhere its all around us, here even in this   room. '
               'You can see it out your window or on your television. '
               'You feel it when you go to work, or go to church or pay your taxes.')
sentences = sent_tokenize(text = text_sample) # 리스트 안에 문장 분리
print(f'타입: {type(sentences)}, 문장 수: {len(sentences)}')
for s in sentences:
  print('-', s)

sentences

타입: <class 'list'>, 문장 수: 3
- The Matrix is everywhere its all around us, here even in this   room.
- You can see it out your window or on your television.
- You feel it when you go to work, or go to church or pay your taxes.


['The Matrix is everywhere its all around us, here even in this   room.',
 'You can see it out your window or on your television.',
 'You feel it when you go to work, or go to church or pay your taxes.']

### 단어 토큰화(Word Tokenization)

[word_tokenize 동작 원리]
- 공백과 구두점을 기준으로 단어 분리
- 구두점(. , !)도 별도 토큰으로 추출됨, 후처리로 구두점 토큰 제거 필요
- it's -> [it , 's]처럼 축약형도 분리

In [ ]:
sentence = 'The Matrix is everywhere its all around us, here even in this   room. '
words = word_tokenize(sentence)
print(f'딘어 수: {len(words)}')
print(words)  # 리스트안에 단어 분리, 구두점 포함

딘어 수: 15
['The', 'Matrix', 'is', 'everywhere', 'its', 'all', 'around', 'us', ',', 'here', 'even', 'in', 'this', 'room', '.']


In [ ]:
import nltk
from nltk import sent_tokenize, word_tokenize

# 문단을 문장별 단어 토큰화 - 문단을 문장 단위로 분리 후 각 문장을 단어 토큰화
def tokenize_text(text):
  sentences = sent_tokenize(text) # 1단계: 문장 분리
  word_tokens = [word_tokenize(s) for s in sentences] # 2단계: 단어 분리
  return word_tokens

word_tokens = tokenize_text(text_sample)
print(f'문장 수: {len(word_tokens)}')
print('첫 번째 문장 단어:', word_tokens[0])

문장 수: 3
첫 번째 문장 단어: ['The', 'Matrix', 'is', 'everywhere', 'its', 'all', 'around', 'us', ',', 'here', 'even', 'in', 'this', 'room', '.']


### 불용어(Stop Words) 제거
[불용어(Stop Words)란?]
- 텍스트 분석에서 의미 없는 매우 빈번한 단어들
- 영어 예시: the, a, an, is, are, was, were, in, at, on, he, she ...
- 제거하면 중요한 내용어(Content Word)에 집중 가능
- 피처 수 감소로 모델 학습 속도 향상
- TF-IDF는 빈출 단어 가중치를 자동 낮추지만 명시적 제거도 효과적

[stopwords.words('english')]
- NLTK가 미리 정의한 179개 영어 불용어 목록
- 언어를 바꾸면 다국어 불용어도 사용 가능: 'french', 'german' 등

In [ ]:
# 불용어(Stop Words) 제거
stopwords = nltk.corpus.stopwords.words('english')
print(f'영어 불용어 수: {len(stopwords)}개')
print('예시 (앞 20개):', stopwords[:20])

영어 불용어 수: 198개
예시 (앞 20개): ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']


In [ ]:
all_tokens = []
for sentence in word_tokens:
  filtered_words = []
  for word in sentence:
    word = word.lower()
    if word not in stopwords:
      filtered_words.append(word)
  all_tokens.append(filtered_words)

print('\n불용어 제거 후 토큰:')
print(all_tokens) #의미어만 남음



불용어 제거 후 토큰:
[['matrix', 'everywhere', 'around', 'us', ',', 'even', 'room', '.'], ['see', 'window', 'television', '.'], ['feel', 'go', 'work', ',', 'go', 'church', 'pay', 'taxes', '.']]


### Stemming(어간 추출)

[Stemming이란?]
- 단어의 어미(또는 일부 접미사)를 규칙에 따라 제거해 어간(stem) 추출
- 사전 없이 규칙 기반으로 처리 -> 빠르지만 부정확할 수 있음
- 어미를 잘라 어근 추정, 단어의 끝 부분을 규칙적으로 잘라내는 과정

[Stemming vs Lemmatization 비교]
- Stemming    : 규칙 기반 어미 제거, 빠름, 결과가 실제 단어 아닐 수 있음
- Lemmatization: 사전 기반 원형 반환, 느림, 항상 실제 단어 반환

[Lancaster Stemmer 특징]
- 가장 공격적인 스테머 (어미를 많이 잘라냄)
- Porter Stemmer보다 더 짧은 어근을 생성
- 장점: 속도 빠름, 피처 수 최소화
- 단점: 과도한 축소로 의미 손실 가능 (fancier→fant)

In [ ]:
from nltk.stem import LancasterStemmer

stemmer = LancasterStemmer()
print('[Stemming 결과]')
print('working/works/worked > ', stemmer.stem('working'), stemmer.stem('works'), stemmer.stem('worked'))
#다양한 형태를 하나의 피처로 묶음

print('amusing/amuses/amused >',
      stemmer.stem('amusing'), stemmer.stem('amuses'), stemmer.stem('amused')) #amuse

print('happier/happiest   >', stemmer.stem('happier'), stemmer.stem('happiest'))

print('fancier/faniest   >', stemmer.stem('fancier'), stemmer.stem('fanciest')) #fant 의미를 힗은 부정확한 어근



[Stemming 결과]
working/works/worked >  work work work
amusing/amuses/amused > amus amus amus
happier/happiest   > happy happiest
fancier/faniest   > fant fanciest


### Lemmatization(표제어 추출): 사전적 원형 반환

[Lemmatization이란?]
- WordNet 사전을 참조해 단어의 기본형(Lemma) 반환
- 항상 실제 사전에 있는 단어를 반환 -> 의미 보존

[품사(pos) 하이퍼 파라미터가 필요한 이유]
- 같은 단어도 품사에 따라 원형이 다를 수 있음
- 예: 'better' → 형용사이면 'good', 동사이면 'better'
- 품사를 지정하지 않으면 기본값(명사)으로 처리 -> 부정확 가능

[품사 태그 약어]
- 'v': verb (동사)
- 'a': adjective (형용사)
- 'n': noun (명사)
- 'r': adverb (부사)

In [ ]:
from nltk.stem import WordNetLemmatizer

lemma = WordNetLemmatizer()
print('[Lemmatization 결과 - 품사 명사 필요]')

print('amusing/amuses/amused >', lemma.lemmatize('amusing', 'v'), lemma.lemmatize('amuses', 'v'), lemma.lemmatize('amused', 'v')) #동사 원형 반환

print('happier/happiest(형용사 a) >', lemma.lemmatize('happier', 'a'), lemma.lemmatize('happiest', 'a')) #형용사 원형 반환

print('fancier/fanciest(형용사 a) >',lemma.lemmatize('fancier', 'a'), lemma.lemmatize('fanciest', 'a')) #stemming보다 의미 보존 우수

[Lemmatization 결과 - 품사 명사 필요]
amusing/amuses/amused > amuse amuse amuse
happier/happiest(형용사 a) > happy happy
fancier/fanciest(형용사 a) > fancy fancy


### BOW - 희소 행렬 (COO · CSR)

[BOW(Bag of Words) 행렬의 특성]
- 행: 문서 수 (예: 25,000개)
- 열: 어휘 수 (예: 100,000개)
- 행렬 크기: 25,000 X 100,000 = 25억 개 원소
- 대부분의 문서는 전체 어휘의 극소수만 사용, 99% 이상이 0
- Dense(밀집) 행렬로 저장하면 메모리 낭비 심각

[희소 행렬 저장 방식]
- 0이 아닌 값(비영값)만 저장 -> 메모리·속도 대폭 개선
- COO: 비영값+행위치+열위치 3개 배열 (직관적, 생성 쉬움)
- CSR: 행 단위로 압축 (ML 연산 최적화, 실무 표준)
- CSC: 열 단위로 압축 (열 방향 연산 최적화)

In [ ]:
import numpy as np
from scipy import sparse
#COO(coordinate) 형식
# 비영값. 해당 값의 행 위치, 열 위치를 각각 배열로 저장

#dense 밀집 행렬 > sparse 희소 행렬 변환
dense = np.array([[3,0,1],
                 [0,2,0]])
data    = np.array([3,1,2]) # 비영값
row_pos = np.array([0,0,1]) # 행 위치
col_pos = np.array([0,2,1]) # 열 위치

# coo__matrix 생성: (값배열, (행위치배열. 열위치배열))
sparse_coo = sparse.coo_matrix((data, (row_pos, col_pos))) #희소 행렬 생성

coo_auto = sparse.coo_matrix((data, (row_pos, col_pos))) #자동 변환
print(sparse_coo.toarray()) # danse 밀집 행렬로 변환
print('원본과 동일:', (dense == sparse_coo.toarray()).all())

#dense (밀집) 행렬: 행렬의 모든 원소(0을 포함한 모든 값)를 저장하는 방식, 일반적인 행력
#Sparse (희소) 행렬: 0이 아닌 값(비영값)과 그 위치 정보만을 저장하는 방식, COO와 CSR 형식

[[3 0 1]
 [0 2 0]]
원본과 동일: True


In [ ]:
# CSR (Compressed sparse Row) 형식 - 행(row) 단위로 데이터를 압축 저장
dense2 = np.array([[0,0,1,0,0,5],
                   [1,4,0,3,2,5],
                   [0,6,0,3,0,0],
                   [2,0,0,0,0,0],
                   [0,0,0,7,0,8],
                   [1,0,0,0,0,0]])
data2         = np.array([1,5,1,4,3,2,5,6,3,2,7,8,1]) #행 순서대로 비영값 나열
col_pos2      = np.array([2,5,0,1,3,4,5,1,3,0,3,5,0]) #각 비영값의 열 위치
row_pos_ind   = np.array([0,2,7,9,10,12,13]) #각 행의 해당하는 값이 data2 배열의 위치를 시작 인덱스. 직전 인덱스 (13)
# 0행:data2[0:2], 1행; data2 [2:7], 2행: data2[7:9], 3행: data2[9:10], 4행:data2[10:12], 5행: data2[12:13]
# 마지막 원소: 전체 비영값 수 (13)

# CSR 직접 생성 (값,열위치,행시작인덱스)
sparse__csr = sparse.csr_matrix((data2, col_pos2, row_pos_ind))

print('csr > Dense 변환:')
print(sparse__csr.toarray())

# 더 간편한 방법: Dense 행렬에서 자동 변환
csr_auto = sparse.csr_matrix(dense2)

print('자동 변환 결과 동일:',(csr_auto.toarray() == dense2).all())

csr > Dense 변환:
[[0 0 1 0 0 5]
 [1 4 0 3 2 5]
 [0 6 0 3 0 0]
 [2 0 0 0 0 0]
 [0 0 0 7 0 8]
 [1 0 0 0 0 0]]
자동 변환 결과 동일: True


### 텍스트 분류 - 20 뉴스그룹

* 20개 카테고리의 뉴스그룹 게시물 약 18,000건
* 텍스트 다중 분류

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 20개 카테고리의 뉴스그룹 게시물 약 18,000건
news_data = fetch_20newsgroups(subset='all', random_state=156) # 학습 + 테스트 전체 로딩
print('키 목록:', news_data.keys())

print('\n카테고리 분포:')
print(pd.Series(news_data.target).value_counts().sort_index())

print('\n카테고리 이름:', news_data.target_names)

키 목록: dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])

카테고리 분포:
0     799
1     973
2     985
3     982
4     963
5     988
6     975
7     990
8     996
9     994
10    999
11    991
12    984
13    990
14    987
15    997
16    910
17    940
18    775
19    628
Name: count, dtype: int64

카테고리 이름: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [ ]:
# 학습/테스트 데이터 분리
train_news = fetch_20newsgroups(subset='train',
                                remove=('headers', 'footers', 'quotes'),
                                random_state=156)
test_news = fetch_20newsgroups(subset='test',
                               remove=('headers', 'footers', 'quotes'),
                               random_state=156)
X_train, y_train = train_news.data, train_news.target
X_test, y_test = test_news.data, test_news.target
print(f'학습: {len(X_train)}건 테스트: {len(X_test)}건')

학습: 11314건 테스트: 7532건


### BOW(Bag of Words)

[CountVectorizer 동작 원리]
- 1. 학습 데이터의 모든 단어로 어휘사전(vocabulary) 구축
- 2. 각 문서를 어휘사전 크기의 벡터로 변환 : 어휘사전에 100개의 고유 단어가 있다면,각 문서는 100개의 숫자로 구성된 벡터로 표현
- 3. 각 위치 값 = 해당 단어의 등장 횟수 (Count)
- 어휘사전에 'cat'(인덱스 0), 'dog'(인덱스 1), 'love'(인덱스 2), 'I'(인덱스 3)가 있다면, 이 문서는 [1, 1, 2, 2, ...] (나머지는 0)와 같은 벡터로 변환

- 결과: (문서 수 x 어휘 수) 이 행렬의 대부분은 0으로 채워지는데, 이는 대부분의 문서가 전체 어휘사전의 극히 일부 단어만을 사용하기 때문. 이를 방지하기 위해 Scikit-learn은 희소행렬 CSR 형식을 사용

In [ ]:
cnt_vect = CountVectorizer()
cnt_vect.fit(X_train) # 학습 데이터로만 어휘사전 구축

X_train_cnt = cnt_vect.transform(X_train) # 학습셋 벡터화
X_test_cnt = cnt_vect.transform(X_test) # 학습 때 만든 어휘사전 그대로 재사용 (테스트셋 단어 무시)

print('CountVectorizer 행렬 shape:', X_train_cnt.shape)

lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_cnt, y_train)
pred = lr_clf.predict(X_test_cnt)
print(f'Count 기반 정확도: {accuracy_score(y_test, pred):.3f}')

CountVectorizer 행렬 shape: (11314, 101631)
Count 기반 정확도: 0.617


### TF-IDF: Count의 한계를 개선한 가중치

[TF-IDF(Term Frequency - Inverse Document Frequency)란?]
- TF(단어 빈도): 해당 문서에서 단어 등장 횟수 (자주 나올수록 중요)
- IDF(역문서 빈도): 전체 문서 중 해당 단어가 등장한 문서 수의 역수, (많은 문서에 등장 -> 덜 중요 -> 낮은 IDF)
- TF-IDF = TF x IDF
- 특정 문서에서만 자주 나오는 단어(핵심어)에 높은 가중치
- 모든 문서에 고르게 나오는 단어(일반어)는 낮은 가중치

[CountVectorizer와의 차이]
- Count: 단순 빈도 -> 'the' 같은 흔한 단어도 높은 값
- TF-IDF: 빈도 x 희귀성 → 문서별 특징 단어에 높은 가중치

In [ ]:
# TF-IDF: 특징 단어에만 높은 가중치
tfidf_vect = TfidfVectorizer()
tfidf_vect.fit(X_train)

X_tr = tfidf_vect.transform(X_train)
X_ts = tfidf_vect.transform(X_test)

lr_clf.fit(X_tr, y_train)
print(f'기본 TF-IDF 정확도: {accuracy_score(y_test, lr_clf.predict(X_ts)):.3f}')

기본 TF-IDF 정확도: 0.678


### N-gram이란?
- 텍스트에서 연속된 N개의 단어 묶음을 하나의 단위로 보는 것
- Unigram (1-gram): 각 단어 하나하나를 특징으로 ('I', 'love', 'this', 'movie')
- Bigram (2-gram): 두 단어의 연속된 묶음을 특징으로 ('I love', 'love this', 'this movie')
- Trigram (3-gram): 세 단어의 연속된 묶음을 특징으로 ('I love this', 'love this movie')

- 예: "not good" -> "not_good"을 피처로 추가 -> 부정 표현 포착
- 장점: 단어 순서 정보 일부 반영
- 단점: 피처 수 급증(unigram의 몇 배)
- N-gram이라는 개념을 사용하여 텍스트 특징(feature)을 어떻게 생성할지 지정

In [ ]:
#옵션 추가
tfidf_vect2 = TfidfVectorizer(stop_words='english', # 불용어 자동 제거
                             ngram_range=(1, 2), #unigram(단어) + bigram(2연속단어)
                             max_df=300 # 300개 이상 문서에 등장하는 단어 제거, 너무 흔한 단어는 특정 카테고리 구분에 도움 안 됨
                             )

tfidf_vect2.fit(X_train)
X_tr2 = tfidf_vect2.transform(X_train)
X_ts2 = tfidf_vect2.transform(X_test)
lr_clf.fit(X_tr2, y_train)
print(f'옵션 TF-IDF 정확도: {accuracy_score(y_test, lr_clf.predict(X_ts2)):.3f}')

옵션 TF-IDF 정확도: 0.690


In [ ]:
# Pipeline: 텍스트 전처리 + 학습을 하나로 통합
pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_df=300)),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10)) # C가 클수록 규제 약함(과적합 가능), 작을수록 규제 강함(과소적합 가능)

])
pipeline.fit(X_train, y_train) #Pipeline 내부에서 TF-IDF 변환 후 LogisticRegression 학습 자동 수행
pred = pipeline.predict(X_test) # 내부에서 TF-IDF 변환 후 예측 자동 수행
print(f'Pipeline 정확도: {accuracy_score(y_test, pred):.3f}')

Pipeline 정확도: 0.704


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.4 MB/s eta 0:00:00


In [ ]:
# Optuna를 이용한 하이퍼파라미터 튜닝(최적화)
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial): # 하이퍼 파라미터 탐색 공간 정의
  ngram_range = trial.suggest_categorical('tfidf_vect__ngram_range', [(1, 1), (1, 2), (1, 3)]) # (min_n, max_n) 형태
  max_df = trial.suggest_categorical('tfidf_vect__max_df', [100, 300, 700])
  C = trial.suggest_categorical('lr_clf__C', [1, 5, 10])

  pipeline = Pipeline([ # 제안된 하이퍼 파라미터로 파이프라인 생성
    ('tfidf_vect', TfidfVectorizer(stop_words='english', ngram_range=ngram_range, max_df=max_df)),
    ('lr_clf', LogisticRegression(solver='liblinear', C=C)) # C가 클수록 규제 약함(과적합 가능), 작을수록 규제 강함(과소적합 가능)
  ])

  # 교차 검증을 통해 파이프라인 성능 평가
  scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1)
  accuracy = scores.mean()

  return accuracy

# Optuna study 생성 및 최적화 실행
study = optuna.create_study(direction='maximize') #'maximize' 방향으로 accuracy를 최적화
study.optimize(objective, n_trials=30, show_progress_bar=True)

print('\n최적 하이퍼 파라미터:', study.best_params)
print(f'최적 CV 스코어: {study.best_value:.3f}' )

[I 2026-04-22 10:42:17,230] A new study created in memory with name: no-name-6aa83876-f0ae-4317-934b-b6e6e3ec4071


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-22 10:44:53,636] Trial 0 finished with value: 0.7503100113149795 and parameters: {'tfidf_vect__ngram_range': (1, 3), 'tfidf_vect__max_df': 300, 'lr_clf__C': 10}. Best is trial 0 with value: 0.7503100113149795.
[I 2026-04-22 10:47:59,456] Trial 1 finished with value: 0.7476583354260092 and parameters: {'tfidf_vect__ngram_range': (1, 3), 'tfidf_vect__max_df': 700, 'lr_clf__C': 5}. Best is trial 0 with value: 0.7503100113149795.
[I 2026-04-22 10:48:24,988] Trial 2 finished with value: 0.7493374442581894 and parameters: {'tfidf_vect__ngram_range': (1, 1), 'tfidf_vect__max_df': 700, 'lr_clf__C': 5}. Best is trial 0 with value: 0.7503100113149795.
[I 2026-04-22 10:49:58,940] Trial 3 finished with value: 0.7368748206696206 and parameters: {'tfidf_vect__ngram_range': (1, 2), 'tfidf_vect__max_df': 300, 'lr_clf__C': 1}. Best is trial 0 with value: 0.7503100113149795.
[I 2026-04-22 10:50:16,710] Trial 4 finished with value: 0.7342235900308572 and parameters: {'tfidf_vect__ngram_range':

In [ ]:
# Optuna가 찾은 최적 하이퍼 파라미터로 최종 파이프라인 구축 및 학습

best_pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english',
                                   ngram_range=study.best_params['tfidf_vect__ngram_range'],
                                   max_df=study.best_params['tfidf_vect__max_df'])),
    ('lr_clf', LogisticRegression(solver='liblinear', C=study.best_params['lr_clf__C'], random_state=156))
])
best_pipeline.fit(X_train, y_train)
pred = best_pipeline.predict(X_test)
print(f'Pipeline 정확도: {accuracy_score(y_test, pred):.3f}')

Pipeline 정확도: 0.704


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

Mounted at /content/drive


### 감성 분석(Sentiment Analysis)

* IMDB 영화 리뷰 감성 분석

In [ ]:
review_df = pd.read_csv('/content/drive/MyDrive/KWU/ML/data/labeledTrainData.tsv',
                        header=0, sep='\t', quoting=3)# 따옴표를 특별 처리 없이 그대로 읽음
print('데이터 shape:', review_df.shape)
print('레이블 분포')
print(review_df['sentiment'].value_counts())
review_df.head(3)

데이터 shape: (25000, 3)
레이블 분포
sentiment
1    12500
0    12500
Name: count, dtype: int64


,id,sentiment,review
0,"""5814_8""",1,"""With all this stuff going down at the moment ..."
1,"""2381_9""",1,"""\""The Classic War of the Worlds\"" by Timothy ..."
2,"""7759_3""",0,"""The film starts with a manager (Nicholas Bell..."


In [ ]:
import re

In [ ]:
# 텍스트 클렌징(Text Cleansing)
# HTML 형식으로 저장 -> 태그 제거
review_df['review'] = review_df['review'].str.replace('<br />', ' ', regex=False) # 정규식이 아닌 문자 그대로 치환

# 숫자, 특수문자는 감성 분석에 노이즈 -> 제거
review_df['review'] = review_df['review'].apply(lambda x: re.sub('[^a-zA-Z]', ' ', x))
# 입력으로 받은 x 문자열에 대해


In [ ]:
from sklearn.model_selection import train_test_split

#학습(70%) / 테스트(30%) 분할
class_df = review_df['sentiment'] # 레이블: 0 부정 또는 1 긍정
feature_df = review_df.drop(['id', 'sentiment'], axis=1) # 피처: 리뷰

X_train, X_test, y_train, y_test = train_test_split(
    feature_df, class_df, test_size=0.3, random_state=156
)
print('학습셋:', X_train.shape, '테스트셋:', X_test.shape)

학습셋: (17500, 1) 테스트셋: (7500, 1)


### [감성 분석(Sentiment Analysis)이란?]
- 텍스트에서 감성(긍정/부정/중립)을 자동으로 판별하는 NLP 태스크
- 이진 분류로 접근: 1=긍정, 0=부정

[지도 학습 감성 분석]
- 레이블(정답)이 있는 데이터로 모델 학습 -> 새 텍스트 감성 예측
- 장점: 정확도 높음 / 단점: 레이블 데이터 필요

In [ ]:
# 지도 학습 감성 분석: Count vs TF-IDF + 로지스틱 회귀
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score

# Count + 로지스틱 회귀
pipeline_cnt = Pipeline([
    ('cnt_vect', CountVectorizer(stop_words='english', ngram_range=(1, 2))),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))
])
pipeline_cnt.fit(X_train['review'], y_train)
pred_cnt = pipeline_cnt.predict(X_test['review'])
prob_cnt = pipeline_cnt.predict_proba(X_test['review'])[:,1] #긍정(클래스 1)확률 -> AUC 계산에 사용
print(f'Count 기반 정확도: {accuracy_score(y_test, pred_cnt):.3f}, AUC: {roc_auc_score(y_test, prob_cnt):.3f}')
# TF-IDF + 로지스틱 회귀
pipeline_tfidf = Pipeline([
    ('tfidf_vect', CountVectorizer(stop_words='english', ngram_range=(1, 2))),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))
])
pipeline_tfidf.fit(X_train['review'], y_train)
pred_tfidf = pipeline_tfidf.predict(X_test['review'])
prob_tfidf = pipeline_tfidf.predict_proba(X_test['review'])[:,1] #긍정(클래스 1)확률 -> AUC 계산에 사용
print(f'Count 기반 정확도: {accuracy_score(y_test, pred_tfidf):.3f}, AUC: {roc_auc_score(y_test, prob_tfidf):.3f}')

Count 기반 정확도: 0.886, AUC: 0.950
Count 기반 정확도: 0.886, AUC: 0.950


### WordNet
- 프린스턴 대학교에서 개발된 영어 어휘의 대규모 관계형 데이터베이스(relational database)
- 단순히 단어의 정의를 제공하는 것을 넘어,
- 단어들 간의 다양한 의미 관계(예: 동의어, 반의어, 상위어/하위어, 부분/전체 관계 등)를 구조적으로 표현하여
- 사람이 단어를 이해하는 방식과 유사하게 설계
- 자연어 처리(NLP) 분야에서 단어의 의미를 파악하고 모호성을 해소하는 데 널리 사용

### [비지도 감성 분석이란?]
- 레이블 데이터(정답) 없이 사전 기반으로 감성 판별
- 장점: 학습 데이터 불필요 -> 도메인 무관하게 바로 적용 가능
- 단점: 지도 학습보다 정확도 낮음, 도메인 특화 표현 포착 어려움

[SentiWordNet이란?]
- WordNet의 각 synset(개념 집합)에 감성 점수를 부여한 어휘 사전
- pos_score: 긍정 점수 (0~1)
- neg_score: 부정 점수 (0~1)
- obj_score: 객관성 점수 (1 - pos - neg) > 0: 긍정 이고 < 0: 부정

[VADER(Valence Aware Dictionary and sEntiment Reasoner)란?]
- 소셜미디어·리뷰 텍스트에 특화된 규칙 기반 감성 분석기
- 어휘사전 + 규칙 기반 -> 학습 데이터 없이 바로 사용 (비지도)

[VADER의 특징]
- 대소문자 구분: "GREAT"는 "great"보다 강한 긍정으로 처리
- 구두점 강조: "great!!!"는 "great"보다 강한 긍정
- 부정 처리: "not good" -> 긍정 단어 good의 점수를 낮춤
- 이모티콘: ":)" 같은 이모티콘도 감성 점수 보유

[SentiWordNet vs VADER 비교]
- SentiWordNet: 각 단어 품사별 점수 -> 정확하지만 느림, 복잡
- VADER: 전체 텍스트를 한 번에 처리 -> 빠름, 소셜 특화

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('sentiwordnet')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package sentiwordnet to /root/nltk_data...
[nltk_data]   Unzipping corpora/sentiwordnet.zip.


True

In [ ]:
from nltk.corpus import wordnet as wn
from nltk.corpus import sentiwordnet as swn
from nltk.stem import WordNetLemmatizer
from nltk import sent_tokenize, word_tokenize, pos_tag # 텍스트 내의 각 단어에 품사 태그를 부여